In [1]:
# Upload df.csv (chargeback data)
from google.colab import files
uploaded = files.upload()

Saving df.csv to df.csv


In [2]:
# Load and engineer features
import pandas as pd
import numpy as np

df = pd.read_csv('df.csv', parse_dates=['Date'])
df = df.drop(columns=['Unnamed: 0'])
df['target'] = (df['CBK'] == 'Yes').astype(int)
df = df.sort_values(['Card Number', 'Date']).reset_index(drop=True)

# How many times has this card transacted in the last 24 hours?
df['tx_count_24h'] = df.groupby('Card Number')['Date'].transform(
    lambda s: s.apply(lambda t: ((s >= t - pd.Timedelta(hours=24)) & (s < t)).sum())
)

# Minutes since this card's previous transaction (-1 if it's the card's first)
df['prev_tx_time'] = df.groupby('Card Number')['Date'].shift(1)
df['minutes_since_last_tx'] = (df['Date'] - df['prev_tx_time']).dt.total_seconds() / 60
df['minutes_since_last_tx'] = df['minutes_since_last_tx'].fillna(-1)

# How unusual is this amount compared to this card's own average spend?
df['card_avg_amount'] = df.groupby('Card Number')['Amount'].transform(
    lambda s: s.expanding().mean().shift(1)
)
df['amount_vs_card_avg'] = (df['Amount'] / df['card_avg_amount'].replace(0, np.nan)) - 1
df['amount_vs_card_avg'] = df['amount_vs_card_avg'].fillna(0)

# Was this transaction at an odd hour (before 6am or after 10pm)?
df['hour'] = df['Date'].dt.hour
df['is_odd_hour'] = ((df['hour'] < 6) | (df['hour'] > 22)).astype(int)

print(df[['Card Number','Date','Amount','target','tx_count_24h',
          'minutes_since_last_tx','amount_vs_card_avg','is_odd_hour']].head(8))
print("\nShape:", df.shape)

df.to_csv('chargeback_features.csv', index=False)
print("Saved chargeback_features.csv")

        Card Number                Date  Amount  target  tx_count_24h  \
0  400217******1137 2015-05-06 09:37:46   198.0       0             0   
1  400217******1353 2015-05-27 23:37:20   172.5       1             0   
2  400217******1353 2015-05-27 23:38:58   172.5       1             1   
3  400217******1353 2015-05-27 23:40:15   172.5       1             2   
4  400217******1353 2015-05-27 23:41:38   172.5       1             3   
5  400217******1353 2015-05-27 23:43:12   172.5       1             4   
6  400217******1353 2015-05-27 23:44:52   172.5       1             5   
7  400217******1353 2015-05-27 23:46:15   172.5       1             6   

   minutes_since_last_tx  amount_vs_card_avg  is_odd_hour  
0              -1.000000                 0.0            0  
1              -1.000000                 0.0            1  
2               1.633333                 0.0            1  
3               1.283333                 0.0            1  
4               1.383333                 0

In [3]:
# Quick sanity check — confirm the features actually separate the two classes
print(df.groupby('target')[['tx_count_24h','minutes_since_last_tx','amount_vs_card_avg','is_odd_hour']].mean())

        tx_count_24h  minutes_since_last_tx  amount_vs_card_avg  is_odd_hour
target                                                                      
0           0.077499             868.292141            0.027238     0.087447
1           2.148601             288.271037            0.153844     0.139860


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, roc_auc_score
import lightgbm as lgb

df = pd.read_csv('chargeback_features.csv', parse_dates=['Date'])

features = ['Amount', 'tx_count_24h', 'minutes_since_last_tx', 'amount_vs_card_avg', 'is_odd_hour']
X = df[features]
y = df['target']

# Split into train/test — test set is what proves that the model works
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
# Train the classifier
# class_weight='balanced' matters here since only ~5% of transactions are chargebacks
model = lgb.LGBMClassifier(
    n_estimators=200, learning_rate=0.05, max_depth=5,
    class_weight='balanced', random_state=42, verbosity=-1
)
model.fit(X_train, y_train)

LGBMClassifier(class_weight='balanced', learning_rate=0.05, max_depth=5,
               n_estimators=200, random_state=42, verbosity=-1)

In [6]:
# Evaluate — this is the "honest metrics" deliverable for the buildathon
probs = model.predict_proba(X_test)[:, 1]
preds = (probs >= 0.5).astype(int)

print("AUC:", round(roc_auc_score(y_test, probs), 4))
print("Precision @0.5:", round(precision_score(y_test, preds), 4))
print("Recall @0.5:", round(recall_score(y_test, preds), 4))

AUC: 0.9035
Precision @0.5: 0.3654
Recall @0.5: 0.6667


In [7]:
# Step 7: Find the cost-optimal threshold
from sklearn.metrics import precision_recall_curve, precision_score, recall_score

avg_chargeback_amt = df[df['target']==1]['Amount'].mean()
FP_COST = 50
FN_COST = avg_chargeback_amt

precisions, recalls, thresholds = precision_recall_curve(y_test, probs)
best_cost, best_t = float('inf'), 0.5
for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds):
    tp = r * y_test.sum()
    fn = y_test.sum() - tp
    fp = (tp / p - tp) if p > 0 else 0
    cost = fp * FP_COST + fn * FN_COST
    if cost < best_cost:
        best_cost, best_t = cost, t

print(f"Optimal threshold: {best_t:.3f}")
preds = (probs >= best_t).astype(int)
print(f"Precision: {precision_score(y_test, preds):.3f}")
print(f"Recall: {recall_score(y_test, preds):.3f}")
print(f"Estimated cost: ₹{best_cost:.2f}")

Optimal threshold: 0.855
Precision: 0.705
Recall: 0.588
Estimated cost: ₹10015.12


In [8]:
# SHAP — explains WHY each transaction was flagged
!pip install shap --quiet
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
for f, s in sorted(zip(features, mean_abs_shap), key=lambda x: -x[1]):
    print(f"{f}: {s:.4f}")

Amount: 0.9556
minutes_since_last_tx: 0.3968
tx_count_24h: 0.3931
is_odd_hour: 0.0877
amount_vs_card_avg: 0.0396


/usr/local/lib/python3.13/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [9]:
# Save the model so FastAPI service can load it
import joblib
joblib.dump(model, 'chargeback_model.pkl')
print("Model saved.")

Model saved.


In [10]:
import json

metrics = {
    "features": features,
    "auc": round(roc_auc_score(y_test, probs), 4),
    "precision": round(precision_score(y_test, preds), 4),
    "recall": round(recall_score(y_test, preds), 4),
    "threshold": round(float(best_t), 4),
    "fp_cost_assumption": FP_COST,
    "fn_cost_assumption": round(float(FN_COST), 2),
    "estimated_total_cost": round(float(best_cost), 2),
    "test_set_size": len(y_test),
}

with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(json.dumps(metrics, indent=2))

{
  "features": [
    "Amount",
    "tx_count_24h",
    "minutes_since_last_tx",
    "amount_vs_card_avg",
    "is_odd_hour"
  ],
  "auc": 0.9035,
  "precision": 0.7053,
  "recall": 0.5877,
  "threshold": 0.8553,
  "fp_cost_assumption": 50,
  "fn_cost_assumption": 183.3,
  "estimated_total_cost": 10015.12,
  "test_set_size": 2226
}


In [11]:
from google.colab import files
files.download('chargeback_model.pkl')
files.download('metrics.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>